## <font color = darkblue>Practice Problem Set 5 - KEY
    
This set of practice problems is to help review performing basic statistics using `scipy` and `statsmodels`.

In [ ]:
pip install openpyxl

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import statsmodels.stats.multitest as smm
import matplotlib.pyplot as plt
from statsmodels.formula.api import ols
import seaborn as sns

In [ ]:
pd.set_option('display.precision', 2)
pd.set_option('display.max_columns',10)



#### <font color=darkblue> Start by importing  the expression-metadata merged spreadsheet, so that you've got something to work with.

In [ ]:
# A way to bring in Data
from google.colab import files
uploaded = files.upload()

In [ ]:
# importing the melanoma dataset
melanoma_log2 = pd.read_excel('melanoma_zerosRemoved_log2transformed_2025.xlsx',index_col = 0)
melanoma_log2.head()

,Sample_geo_accession,Stage,cell type,sample_number,cell_line,...,ZYX,ZZEF1,ZZZ3,BP-21201H5.1,BP-2189O9.2
Sample Title,,,,,,,,,,,
FM_1,GSM2344965,primary melanocytes,normal melanocytes,1,FM,...,11.66,11.83,10.55,3.17,1.00
FM_2,GSM2344966,primary melanocytes,normal melanocytes,2,FM,...,11.63,11.54,11.06,2.81,2.00
FM_3,GSM2344967,primary melanocytes,normal melanocytes,3,FM,...,12.31,11.44,10.81,2.32,1.58
SK_MEL_28_1,GSM2344968,metastatic,melanoma cell line,4,SK_MEL_28,...,11.86,10.94,11.64,0.00,6.04
SK_MEL_28_2,GSM2344969,metastatic,melanoma cell line,5,SK_MEL_28,...,11.97,11.03,11.74,1.00,6.41


Extract out the gene expression data specific to the different cell lines (one variable per cell line type).

In [ ]:
# Extracting out only the gene expression dat from each of the cell line samples
FMexp = melanoma_log2.loc[melanoma_log2.cell_line == 'FM','A1BG':]
SK28exp = melanoma_log2.loc[melanoma_log2.cell_line == 'SK_MEL_28','A1BG':]
SK147exp = melanoma_log2.loc[melanoma_log2.cell_line == 'SK_MEL_147','A1BG':]
UACCexp = melanoma_log2.loc[melanoma_log2.cell_line == 'UACC_62','A1BG':]

In [ ]:
SK147exp
SK28exp

,A1BG,A2M,AAAS,AACS,AADAT,...,ZYX,ZZEF1,ZZZ3,BP-21201H5.1,BP-2189O9.2
Sample Title,,,,,,,,,,,
SK_MEL_28_1,8.37,11.37,10.76,10.70,8.56,...,11.86,10.94,11.64,0.0,6.04
SK_MEL_28_2,8.50,11.40,10.91,10.74,8.84,...,11.97,11.03,11.74,1.0,6.41
SK_MEL_28_3,8.61,11.48,10.93,10.51,8.50,...,11.94,11.02,11.59,0.0,5.91


Then, calculate the overall variance, sort it in descending order, and extract gene names for the top 10 most variably expressed genes.

In [ ]:
# calculates the overall variance df.var() and sorts it in descending order
overall_variance = melanoma_log2.loc[:,'A1BG':].var()
overall_variance.sort_values(inplace = True, ascending= False)
overall_variance.head()

# extract gene names for top 10 most variably expressed genes
topvarGens = overall_variance.index[:10]
topvarGens

Index(['PMEL', 'TYRP1', 'AEBP1', 'GLUL', 'TYR', 'EEF1A2', 'CDC42EP1', 'A2M',
       'SOD3', 'TGFBI'],
      dtype='object')

### <font color = blue> Correlation:

1. Compare two metastatic cell lines (e.g. SK_MEL_28_1 and SK_MEL_147_1) using the Pearson's correlation test.

In [ ]:
corrP, pP = stats.pearsonr(SK28exp.iloc[0,:], SK147exp.iloc[0,:])
corrP
pP

np.float64(0.0)

In [ ]:
corrP, pP = stats.pearsonr(SK28exp.iloc[0,:], SK147exp.iloc[0,:])

print("the R2 is:"), print(corrP)
print("the p value is:"), print(pP)

the R2 is:
0.8500097727484631
the p value is:
0.0


(None, None)

2. Compare another two metastatic cells lines (e.g. SK_MEL_147_1 and UACC_62_1) using Spearman's rank correlation test.

In [ ]:
corrS, pS = stats.spearmanr(SK147exp.iloc[0,:], UACCexp.iloc[0,:])
print("the R2 is"), print(corrS)
print("the p value is"), print(pS)

the R2 is
0.8510824016760983
the p value is
0.0


(None, None)

3. Calculate the pair-wise correlations of the normal cells' expression data frame using the Pearson method.

In [ ]:
FMexp.transpose().corr(method='pearson')

Sample Title,FM_1,FM_2,FM_3
Sample Title,,,
FM_1,1.00,0.97,0.97
FM_2,0.97,1.00,0.97
FM_3,0.97,0.97,1.00


4. Find the Spearman correlation between each pairwise comparison of the top five most variable genes in the data frame: `melanoma_log2`.

In [ ]:
melanoma_log2.loc[:,topvarGens[:5]].corr(method='spearman')

,PMEL,TYRP1,AEBP1,GLUL,TYR
PMEL,1.00,0.93,0.77,0.39,0.97
TYRP1,0.93,1.00,0.65,0.42,0.94
AEBP1,0.77,0.65,1.00,0.40,0.78
GLUL,0.39,0.42,0.40,1.00,0.37
TYR,0.97,0.94,0.78,0.37,1.00


### <font color = blue> Linear Models for Comparing Groups </font>:

1. Print out a data frame showing two columns: one with the cell line information, and the other with the expression of the gene TYRP1 (second most variably expressed):

In [ ]:
melanoma_log2.loc[:,['cell_line','TYRP1']]

,cell_line,TYRP1
Sample Title,,
FM_1,FM,19.46
FM_2,FM,19.61
FM_3,FM,19.34
SK_MEL_28_1,SK_MEL_28,10.54
SK_MEL_28_2,SK_MEL_28,10.10
SK_MEL_28_3,SK_MEL_28,10.18
SK_MEL_147_1,SK_MEL_147,3.91
SK_MEL_147_2,SK_MEL_147,3.70
SK_MEL_147_3,SK_MEL_147,5.13


2. Test whether there's an association between the value of TYRP1 and the cell line type with a linear regression model

In [ ]:
model = ols("TYRP1 ~ C(cell_line) + 1", melanoma_log2).fit() #C() tells python to treat this as a categorical variable.

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  TYRP1   R-squared:                       0.991
Model:                            OLS   Adj. R-squared:                  0.988
Method:                 Least Squares   F-statistic:                     302.9
Date:                Fri, 28 Mar 2025   Prob (F-statistic):           1.42e-08
Time:                        23:33:46   Log-Likelihood:                -9.9663
No. Observations:                  12   AIC:                             27.93
Df Residuals:                       8   BIC:                             29.87
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           

/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:418: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=12 observations were given.
  return hypotest_fun_in(*args, **kwds)
